> 在线讲义：[第 7 章 · 价值函数 value function：v_π(s) 和 q_π(s, a)](https://wikieden.github.io/rl-from-scratch/lessons/07-value.html) · 本书《从零学强化学习》

# 第 7 章 · 价值函数 value function：v_π(s) 和 q_π(s, a)

本章回答：一个状态"好不好"怎么用一个数说清楚？在某个状态先做某个动作，又值多少？这两个数之间什么关系？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/07-value.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 跑 2000 次估 $v_\pi$ | `mc_value(P, pi, s)` | 高电 15.84，低电 13.89 |
| 第一步固定估 $q_\pi$ | `mc_value(P, pi, 1, first_action=2)` / `=0` | 充电 14.22，搜索 13.46 |
| 精确值（第 8 章） | `ch08_bellman.solve_v`、`q_from_v` | 13.869、14.245、13.511 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `ROBOT_PI`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_PI = {0: {0: 0.9, 1: 0.1}, 1: {0: 0.2, 1: 0.3, 2: 0.5}}

**常量 `ROBOT_STATES`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_STATES = ["高电", "低电"]

**常量 `GAMMA`**（来自 `ch07_value.py`）

In [ ]:
GAMMA = 0.9

**函数 `sample_action`**（来自 `ch06_policy.py`）：下面要用到的小工具。

In [ ]:
def sample_action(pi, s, rng):
    acts = list(pi[s])
    return acts[rng.choice(len(acts), p=[pi[s][a] for a in acts])]

**函数 `step`**（来自 `ch07_value.py`）：下面要用到的小工具。

In [ ]:
def step(P, s, a, rng):
    outs = P[s][a]
    p, ns, r, done = outs[rng.choice(len(outs), p=[o[0] for o in outs])]
    return ns, r

**函数 `mc_value`**（来自 `ch07_value.py`）：从 s0 出发跑 n 次（每次截到 horizon 步，γ^100 ≈ 3e-5 可忽略），把折扣回报取平均。

In [ ]:
def mc_value(P, pi, s0, n_episodes=2000, horizon=100, gamma=GAMMA, seed=0, first_action=None):
    """从 s0 出发跑 n 次（每次截到 horizon 步，γ^100 ≈ 3e-5 可忽略），把折扣回报取平均。"""
    rng, total = np.random.default_rng(seed), 0.0
    for _ in range(n_episodes):
        s, G, disc = s0, 0.0, 1.0
        for t in range(horizon):
            a = first_action if (t == 0 and first_action is not None) else sample_action(pi, s, rng)
            s, r = step(P, s, a, rng)
            G += disc * r
            disc *= gamma
        total += G
    return total / n_episodes

**常量 `ALPHA`, `BETA`**（来自 `rl_envs.py`）

In [ ]:
ALPHA, BETA = 0.8, 0.7

**常量 `R_SEARCH`, `R_WAIT`, `R_RESCUE`**（来自 `rl_envs.py`）

In [ ]:
R_SEARCH, R_WAIT, R_RESCUE = 2.0, 1.0, -3.0

**函数 `robot_mdp`**（来自 `rl_envs.py`）：返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。

In [ ]:
def robot_mdp():
    """返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。"""
    hi, lo = 0, 1
    P = {
        hi: {
            0: [(ALPHA, hi, R_SEARCH, False), (1 - ALPHA, lo, R_SEARCH, False)],
            1: [(1.0, hi, R_WAIT, False)],
        },
        lo: {
            0: [(BETA, lo, R_SEARCH, False), (1 - BETA, hi, R_RESCUE, False)],
            1: [(1.0, lo, R_WAIT, False)],
            2: [(1.0, hi, 0.0, False)],
        },
    }
    return P

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
P, pi = robot_mdp(), ROBOT_PI

In [ ]:
for s in pi:
        print(f"v_π({ROBOT_STATES[s]}) ≈", round(mc_value(P, pi, s), 2))

In [ ]:
print("q_π(低电, 充电) ≈", round(mc_value(P, pi, 1, first_action=2), 2))

In [ ]:
print("q_π(低电, 搜索) ≈", round(mc_value(P, pi, 1, first_action=0), 2))

## 3. 代码里才看得到的

持续任务没有终点，代码只能"截到 100 步"。因为 $0.9^{100} \approx 3 \times 10^{-5}$，截掉的部分最多影响 $2 \times 3\times10^{-5} / 0.1 \approx 0.0005$，可以忽略。γ 越接近 1，要截的步数越多。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. `mc_value(P, pi, 0, n_episodes=20, seed=0 / 1)` → 15.65、15.37；`n_episodes=200` → 15.73、15.88。精确值是 15.828（第 8 章）。局数少时换个种子差得多，局数多时稳得多。
2. `mc_value(P, pi, 0, gamma=0.5)` → 3.483：γ 变小，远处的奖励几乎不算，价值一下就小了。第 8 章用方程解出来是 3.49。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 价值是房价：从这出发按 π 走，折扣回报的平均；
> q 先做 a 再按 π，挑动作只看 q；
> v 是 q 按 π 的平均，q 是"眼前 + γ × 下一个 v"的平均；
> 不会算就多跑几次取平均。